In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
def evaluate_models(T):

    # ar1

    rho = np.array([0.2, 0.8])
    sigma = np.array([0.5, 1.0])
    P = np.array([[0.95, 0.05], [0.05, 0.95]])

    states = np.zeros(T, dtype=int)
    y = np.zeros(T)

    states[0] = np.random.choice([0,1])
    y[0] = np.random.normal()

    for t in range(1, T):
        states[t] = np.random.choice([0,1], p=P[states[t-1]])
        y[t] = rho[states[t]] * y[t-1] + np.random.normal(scale=sigma[states[t]])

    # train-test split

    split = int(T * 0.8)
    y_train, y_test = y[:split], y[split:]

    rho_base = np.sum(y_train[1:] * y_train[:-1]) / np.sum(y_train[:-1]**2)

    # forecasting on out-of-sample

    preds_base, preds_hard, preds_soft = [], [], []

    for i in range(len(y_test)):
        t = split + i
        y_prev = y[t-1]

        p1 = P[states[t-1], 1]
        p0 = P[states[t-1], 0]

        preds_base.append(rho[1]*y_prev)

        if p1 > 0.5:
            preds_hard.append(rho[1] * y_prev)
        else:
            preds_hard.append(rho[0] * y_prev)
        
        preds_soft.append((p0 * rho[0] * y_prev) + (p1 * rho[1] * y_prev))

    rmse_base = np.sqrt(np.mean((y_test - np.array(preds_base))**2))
    rmse_hard = np.sqrt(np.mean((y_test - np.array(preds_hard))**2))
    rmse_soft = np.sqrt(np.mean((y_test - np.array(preds_soft))**2))

    return rmse_base, rmse_hard, rmse_soft 



In [3]:
np.random.seed(42)

sample_sizes = [100, 500, 1000, 5000, 10000]

print(f"{'Sample Size':<15} | {'AR(1) Baseline':<15} | {'HMM Hard':<15} | {'HMM Soft':<15}")
print("-" * 65)

for T in sample_sizes:
    r_base, r_hard, r_soft = evaluate_models(T)
    print(f"{T:<15} | {r_base:<15.4f} | {r_hard:<15.4f} | {r_soft:<15.4f}")

Sample Size     | AR(1) Baseline  | HMM Hard        | HMM Soft       
-----------------------------------------------------------------
100             | 0.8905          | 0.8905          | 0.8915         
500             | 0.7405          | 0.7215          | 0.7359         
1000            | 0.8863          | 0.8494          | 0.8483         
5000            | 0.8110          | 0.7903          | 0.7883         
10000           | 0.8090          | 0.7810          | 0.7797         
